# 02 — Run a Minimal End-to-End Experiment

This notebook exercises the **full pipeline** in miniature: build a coreset with NSGA-II (k=30, small population and few generations), save it, then evaluate it with the standalone metric script. The goal is to confirm your installation works end-to-end and to demystify the data flow before scaling up.

**Prerequisite:** `python scripts/launchers/build_caches.py --reps 0` has been run.

**Runtime:** 2–4 minutes on a laptop (most of it is the NSGA-II run).

**Entry points exercised:**
- [`adaptive_tau.py`](../docs/api/scripts.md#adaptive_tau)
- [`evaluate_coresets.py`](../docs/api/scripts.md#evaluate_coresets)

In [ ]:
import sys, subprocess
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'coreset_selection').is_dir():
    REPO = REPO.parent
print(f'Repo root: {REPO}')

# Quick-run settings (small values for speed)
K = 30
REP_ID = 0
OUTPUT_DIR = REPO / 'runs_out_quickstart'
CACHE_DIR = REPO / 'experiments' / 'adaptive_tau_k100_ps_vae' / 'replicate_cache'

print('K        =', K)
print('Rep      =', REP_ID)
print('Cache    =', CACHE_DIR)
print('Output   =', OUTPUT_DIR)

## 1. Run NSGA-II with adaptive tau

We shell out to the `adaptive_tau.py` CLI. For programmatic use, you could also import `scripts.launchers.adaptive_tau` and call its main function directly — the CLI is simpler for a tutorial.

In [ ]:
cmd = [
    sys.executable, str(REPO / 'scripts' / 'launchers' / 'adaptive_tau.py'),
    '--k', str(K),
    '--space', 'vae',
    '--constraint-mode', 'popsoft',
    '--rep', str(REP_ID),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print('Return code:', res.returncode)
print('Last lines of stdout:')
for line in res.stdout.splitlines()[-10:]:
    print(' ', line)

## 2. Inspect what was saved

The experiment folder follows a predictable structure.

In [ ]:
exp_dir = OUTPUT_DIR / f'nsga2-vae-popsoft-k{K}-rep{REP_ID}'
print(f'Experiment folder: {exp_dir}')
for p in sorted(exp_dir.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(exp_dir)}')

## 3. Peek at the manifest and the Pareto front

In [ ]:
import json, numpy as np

with open(exp_dir / 'manifest.json') as f:
    manifest = json.load(f)
print('Manifest summary:')
for key in ('rep_id', 'seed', 'k', 'space', 'constraint_mode', 'objectives', 'tau_final', 'n_pareto_solutions'):
    print(f'  {key:<22}: {manifest.get(key)}')

with np.load(exp_dir / 'coreset.npz', allow_pickle=True) as data:
    X = data['X']
    F = data['F']
    objectives = list(data['objectives'])
print(f'\nPareto front: X shape {X.shape}, F shape {F.shape}, objectives = {objectives}')

## 4. Evaluate all Pareto-front members

In [ ]:
cache_path = CACHE_DIR / f'rep{REP_ID:02d}' / 'assets.npz'
cmd = [
    sys.executable, str(REPO / 'scripts' / 'analysis' / 'evaluate_coresets.py'),
    '--experiment-dir', str(exp_dir),
    '--cache-path', str(cache_path),
]
res = subprocess.run(cmd, capture_output=True, text=True)
print('Return code:', res.returncode)
print('Last lines:')
for line in res.stdout.splitlines()[-6:]:
    print(' ', line)

## 5. Read `metrics.csv`

In [ ]:
import pandas as pd
df = pd.read_csv(exp_dir / 'metrics.csv')
print(f'Rows: {len(df)}  Columns: {df.shape[1]}')
key_cols = [c for c in ('coreset_name', 'k', 'nystrom_error', 'kpca_distortion') if c in df.columns]
df[key_cols].head(10)

## What's next

- [03_visualize_pareto_front.ipynb](./03_visualize_pareto_front.ipynb) — plot the Pareto front.
- [04_interpret_metrics.ipynb](./04_interpret_metrics.ipynb) — understand every metric column.